# LangGraph入门笔记
## 1. 为什么需要LangGraph
对于普通的LLM调用可能是如下的流程；
<br>用户输入 -> 调用模型 -> 返回结果
<br>但是对于真实的AI应用来说，可能会有复杂的工作流程，如：
<p>用户提问
 -> 判断是否需要搜索
 -> 如果需要，调用工具
 -> 整理工具结果
 -> 再调用模型生成回答
 -> 如果不满意，继续反思或重试
 -> 最终返回结果
 </p>

这种流程有几个难点：
- 步骤多
- 需要保存中间状态
- 可能有分支
- 可能有循环
- 可能需要人工介入
- 可能需要失败后恢复

<p>LangGraph 就是为这些场景设计的。官方也把它定位为用于构建、管理、部署 long-running
stateful agents 的低层编排框架，也就是“长时间运行、有状态的智能体编排框架”。</p>

LangGraph的核心思想：**用“图”来组织 AI 应用流程的框架。**

### LangGraph的核心概念
State：Agent的共享上下文，（通常由一个tydedict定义）
Node：每个节点代表一个处理步骤（通常由一个函数表示），例如调用LLM、调用工具、检索知识库、人工审核
Edge：表示各步骤之间的关系
Conditional Edge = 根据状态决定下一步去哪
Graph = 整个Agent应用的执行流程

一句话总结：**LangGraph = 用 State 保存流程数据，用 Node 处理数据，用 Edge 控制执行顺序。**
### 一个典型的Agent流程图
```text
用户输入
  ↓
LLM 思考
  ↓
是否需要工具？
  ├── 是 → 调用工具 → 把结果写回 State → 再让 LLM 思考
  └── 否 → 输出最终答案
```

## 2. 最小LangGraph工作流
输入一个字符串（State），经过两个Node处理<br>
start → add_hello → add_suffix → end


In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END

class State(TypedDict):
    text:str

def add_hello(state:State) -> State:
    return {
        "text":"Hello" + state['text']
    }

def add_suffix(state:State) -> State:
    return{
        "text": state['text'] + "!"
    }

builder = StateGraph(State)
builder.add_node("add_hello" , add_hello)
builder.add_node("add_suffix" , add_suffix)

builder.add_edge(START , "add_hello")
builder.add_edge('add_hello' , "add_suffix")
builder.add_edge("add_suffix",END)

graph = builder.compile()
result = graph.invoke({'text' : "LangGraph"})
print(result)

### 语法总结
1. 状态定义class State：使用Python原生的typing_extensions.TypedDict来定义图的状态Schema
2. 节点函数：定义函数，接收当前状态 (state) 作为参数，并返回一个字典。返回的字典会自动与当前状态进行合并/更新（在这个 TypedDict 的默认行为下，同名键的值会被覆盖替换）
3. builder = StateGraph(State)：实例化状态图，是LangGraph提供的最主要的图构建器
4. builder.add_node("add_hello", add_hello)：将定义好的函数注册到图中，add_node(node_name, action)。第一个参数是节点的名称（字符串标识符，供后续连线使用），第二个参数是具体的执行函数。
5. builder.add_edge(START, "add_hello")、builder.add_edge("add_suffix", END)：<br>定义节点是如何流动的，即行为是如何配合的，**START**和**END**标志了图的入口和出口
6. graph = builder.compile()：将构建好的拓扑结构（节点和边）“冻结”并编译成一个可运行的 CompiledGraph（本质上是一个 Runnable 对象）。在这一步，LangGraph 会进行简单的图校验（比如是否有成环死循环、是否没有连向 END 等）

## 2. StateGraph的更新状态机制
LangGraph里很多初学者的bug都来自一个误解：以为节点函数是在“修改 state”，但实际上节点函数是在“返回 state 的更新”。<br>
通过下面一段代码来说明这种差别
```python
class State(TypedDict):
    name: str
    count: int

def wrong_node(state: State) -> dict:
    state["count"] += 1
    return state
```
这段代码实际上是节点原地修改全局变量
```python
def correct_node(state: State) -> dict:
    return {"count": state["count"] + 1}
```
而这段代码则是将State进行更新，把新字段count覆盖旧的count
这也就意味着StateGraph 的节点会读取共享State，并返回Partial<State>，也就是状态的一部分更新；每个state key也可以配置reducer来决定多个更新如何聚合。
所以说LangGraph的State更新有以下几个特点：
1. State是所有节点共享d
2. Node返回的是状态更新，即return {"字段名": 新值}
3. Node当中没更新的字段会得到保留
4. 默认情况下，同名字段会被覆盖，但是LangGraph会使用Reducer来完成“追加、累加、合并”

## 3.Reducer 与列表累积机制
由前面内容我们知道：默认情况下，**节点返回的新字段值会覆盖旧字段值**。但是在某些场景下Agent场景不是想覆盖，而是想累积。比如对话记忆等等。这时候就需要Reducer。
### Reducer是什么？
一句话解释：当多个节点或多次更新同一个字段时，LangGraph应该如何合并旧值和新值。

### 先看不使用Reducer的情况

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    messages: list[str]


def user_node(state: State) -> dict:
    return {"messages": ["用户：你好"]}


def ai_node(state: State) -> dict:
    return {"messages": ["AI：你好，我是 LangGraph 助手"]}


builder = StateGraph(State)

builder.add_node("user_node", user_node)
builder.add_node("ai_node", ai_node)

builder.add_edge(START, "user_node")
builder.add_edge("user_node", "ai_node")
builder.add_edge("ai_node", END)

graph = builder.compile()

result = graph.invoke({"messages": []})

print(result)
#{'messages': ['AI：你好，我是 LangGraph 助手']}

### 使用Reducer：让messages自动追加
在Python里，常见的写法是用Annotated给字段加一个合并函数。

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    messages: Annotated[list[str], add_messages]
'''
messages: Annotated[list[str], add]——这一行告诉LangGraph当messages被更新时，不要覆盖，而是用operator.add合并。
即对于列表来说，使用add = ["用户：你好"] + ["AI：你好"]，得到["用户：你好", "AI：你好"]
'''

def user_node(state: State) -> dict:
    return {"messages": ["用户：你好"]}


def ai_node(state: State) -> dict:
    return {"messages": ["AI：你好，我是 LangGraph 助手"]}


builder = StateGraph(State)

builder.add_node("user_node", user_node)
builder.add_node("ai_node", ai_node)

builder.add_edge(START, "user_node")
builder.add_edge("user_node", "ai_node")
builder.add_edge("ai_node", END)

graph = builder.compile()

result = graph.invoke({"messages": []})

print(result)

## 4. 条件分支（Conditional Edge）
在真实场景中，Agent经常需要判断，比如：<br>
用户问天气 -> 调用天气工具<br>
用户问翻译 -> 调用翻译节点<br>
用户问普通问题 -> 直接回答<br>
因此需要引入条件分支，而conditional_edges就是让LangGraph能根据State内容，决定下一个去哪个节点。<br>
以下是一个“START -> classify_node -> 根据分类结果选择后续节点”的示例

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    question: str
    intent: str
    answer: str


def classify_node(state: State) -> dict:
    q = state["question"]

    if "天气" in q:
        return {"intent": "weather"}
    elif "翻译" in q:
        return {"intent": "translate"}
    else:
        return {"intent": "chat"}


def route_by_intent(state: State) -> str:
    return state["intent"]


def weather_node(state: State) -> dict:
    return {"answer": "这里应该调用天气工具。"}


def translate_node(state: State) -> dict:
    return {"answer": "这里应该调用翻译工具。"}


def chat_node(state: State) -> dict:
    return {"answer": "这里直接让模型回答普通问题。"}


builder = StateGraph(State)

builder.add_node("classify_node", classify_node)
builder.add_node("weather_node", weather_node)
builder.add_node("translate_node", translate_node)
builder.add_node("chat_node", chat_node)

builder.add_edge(START, "classify_node")

builder.add_conditional_edges(
    "classify_node",
    route_by_intent,
    {
        "weather": "weather_node",
        "translate": "translate_node",
        "chat": "chat_node",
    }
)

builder.add_edge("weather_node", END)
builder.add_edge("translate_node", END)
builder.add_edge("chat_node", END)

graph = builder.compile()

print(graph.invoke({
    "question": "今天上海天气怎么样？",
    "intent": "",
    "answer": ""
}))

print(graph.invoke({
    "question": "请帮我翻译 hello",
    "intent": "",
    "answer": ""
}))

print(graph.invoke({
    "question": "LangGraph 是什么？",
    "intent": "",
    "answer": ""
}))

'''输出
{
    "question": "今天上海天气怎么样？",
    "intent": "weather",
    "answer": "这里应该调用天气工具。"
}

{
    "question": "请帮我翻译 hello",
    "intent": "translate",
    "answer": "这里应该调用翻译工具。"
}

{
    "question": "LangGraph 是什么？",
    "intent": "chat",
    "answer": "这里直接让模型回答普通问题。"
}
'''

以该输入为例：
```python
{
    "question": "今天上海天气怎么样？",
    "intent": "",
    "answer": ""
}
```
第一步进入classify_node:
```python
if "天气" in q:
    return {"intent": "weather"}
```
状态更新为：
```python
{
    "question": "今天上海天气怎么样？",
    "intent": "weather",
    "answer": ""
}
```
然后执行条件路由函数：
```python
def route_by_intent(state: State) -> str:
    return state["intent"]
```
返回
```python
"weather"
```
映射表里
```python
"weather": "weather_node"
```
所以最终进入weather_node
```python
return {"answer": "这里应该调用天气工具。"}
```
最终结果
```python
{
    "question": "今天上海天气怎么样？",
    "intent": "weather",
    "answer": "这里应该调用天气工具。"
}
```

## 5. 循环Loop
在前面的章节中，我们已经学会了让Agent具有判断能力，但是在真实的场景中，Agent经常不是一次性就能完成一个任务，而是需要多次的循环，才能完成一个任务，比如
```text
模型思考
 -> 调用工具
 -> 工具返回结果
 -> 模型再次思考
 -> 如果还需要工具，继续调用
 -> 如果不需要工具，结束
```
所以，LangGraph提供了循环机制来支持这种多步流程。通过使用循环，Agent可以反复执行思考-调用工具-获取结果的步骤，直到任务完成。
### 5.1 最小循环示例

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    count: int


def add_one(state: State) -> dict:
    return {"count": state["count"] + 1}


def should_continue(state: State) -> str:
    if state["count"] < 3:
        return "continue"
    else:
        return "end"


builder = StateGraph(State)

builder.add_node("add_one", add_one)

builder.add_edge(START, "add_one")

builder.add_conditional_edges(
    "add_one",
    should_continue,
    {
        "continue": "add_one",
        "end": END,
    }
)

graph = builder.compile()

result = graph.invoke({"count": 0})

print(result)

### 5.2 循环代码解读
```python
....
def should_continue(state: State) -> str:
    if state["count"] < 3: # 边界条件
        return "continue"
    else:
        return "end"

builder.add_conditional_edges(
    "add_one",
    should_continue, # 条件分支
    {
        "continue": "add_one",
        "end": END,
    }
)
....
```
在上述代码中，我们可以看出，通过判断当前state的状态来判断是否达到设定的边界值，如果达到边界值，则返回"end"，否则执行相应的行为，这和传统的while循环的核心思路是一致。在LangGraph中，采用条件分支去控制每一步循环体执行完后，是否要推出循环。

在循环Loop当中，最容易犯的错误是**忘记终止条件**。即使LangGraph 通常会有递归/步数限制保护，但你不能依赖它。写循环时**必须有清晰的停止条件**。

在引入了循环Loop之后，一个利用工具的循环Agent就可以设计为：；
